# Import Libraries

In [ ]:
# --------------------------------------------
# 0. Imports & global settings
# --------------------------------------------
import sys
import os
from pathlib import Path

# Set working directory to project root, if not done already.
project_root = Path('/Users/raymondlow/Documents/talking-to-machines/ai-population').resolve()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Set __package__ so that relative imports work.
__package__ = "ai_population.analysis"

import os, re
import numpy as np
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from langdetect import detect
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2, mutual_info_classif

NLP = spacy.load("en_core_web_sm", disable=["ner", "parser"])  # fast pipeline
URL_RE = re.compile(r"http\S+|www\.\S+")
HASHTAG_OR_CASHTAG_RE = re.compile(r"(#\w+|\$\w+)")
TOKEN_PATTERN = r"(?u)(?:#\w+|\$\w+|\b\w+\b)"  # CountVectorizer regex

# Set display options to show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

platform = "x"  # tiktok or x
PROJECT_NAME = f"market-signals-{platform}"
EXECUTION_DATE = "ground-truth"
TOP_N_TERMS = 30

# Identify Relevant Search Terms from Validation Set

In [ ]:
# --------------------------------------------
# 1. Helper functions
# --------------------------------------------
def is_english(txt):
    try:
        return detect(txt) == "en"
    except Exception:
        return False

def preprocess(text):
    """
    • Lower-case
    • Strip URLs
    • Keep hashtags & cashtags
    • Lemmatise (spaCy) except hashtags/cashtags
    Returns cleaned string.
    """
    # Remove URLs
    text = URL_RE.sub(" ", text.lower())

    # Extract hashtags/cashtags intact
    preserved = HASHTAG_OR_CASHTAG_RE.findall(text)

    # Remove them from text for lemmatization
    text_wo_tags = HASHTAG_OR_CASHTAG_RE.sub(" ", text)
    doc = NLP(text_wo_tags)
    lemmas = [t.lemma_ for t in doc if not t.is_stop and not t.is_punct and t.lemma_.strip()]

    return " ".join(lemmas + preserved)

def combine_tiktok_metadata_text(row: pd.Series) -> str:
    combined_text_list = []

    # Append post description
    if row["description"] is not None and not pd.isnull(row["description"]):
        combined_text_list.append(row["description"].replace("\n\n","\n"))

    # Append post transcript
    if row["video_transcript"] is not None and not pd.isnull(row["video_transcript"]):
        combined_text_list.append(row["video_transcript"].replace("\n\n","\n"))
        
    if combined_text_list == []:
        return ""
    else:
        return preprocess("\n".join(combined_text_list))
    
def combine_x_metadata_text(row: pd.Series) -> str:
    combined_text_list = []

    # Append post text
    if row["text"] is not None and not pd.isnull(row["text"]):
        combined_text_list.append(row["text"].replace("\n\n","\n"))
        
    if combined_text_list == []:
        return ""
    else:
        return preprocess("\n".join(combined_text_list))

# --------------------------------------------
# 2. Load & clean data
# --------------------------------------------
# Load validation finfluencer and non-finfluencer lists
validation_profiles = pd.read_csv(os.path.join("ai_population/data", PROJECT_NAME, EXECUTION_DATE, "ground_truth_validation_set.csv"))
ground_truth_finfluencers = validation_profiles[validation_profiles["finfluencer"] == 1]["account_id"].tolist()
ground_truth_nonfinfluencers = validation_profiles[validation_profiles["finfluencer"] == 0]["account_id"].tolist()
print(f"Number of Financial Influencers: {len(ground_truth_finfluencers)}")
print(f"Number of Non-Financial Influencers: {len(ground_truth_nonfinfluencers)}")

# Split posts into finfluencer and non-finfluencer categories
post_data = pd.read_csv(os.path.join("ai_population/data", PROJECT_NAME, EXECUTION_DATE, "ground_truth_profile_posts.csv"))
finfluencer_post_data = post_data[post_data["account_id"].isin(ground_truth_finfluencers)].reset_index(drop=True)
finfluencer_post_data["finfluencer"] = 1
nonfinfluencer_post_data = post_data[post_data["account_id"].isin(ground_truth_nonfinfluencers)].reset_index(drop=True)
nonfinfluencer_post_data["finfluencer"] = 0

# Combine profile metadata and post metadata for each post
if platform == "tiktok":
    finfluencer_post_data['combined_text'] = finfluencer_post_data.apply(combine_tiktok_metadata_text, axis=1)
    finfluencer_post_data["cleaned_text"] = finfluencer_post_data["combined_text"].astype(str).apply(preprocess)
    nonfinfluencer_post_data['combined_text'] = nonfinfluencer_post_data.apply(combine_tiktok_metadata_text, axis=1)
    nonfinfluencer_post_data["cleaned_text"] = nonfinfluencer_post_data["combined_text"].astype(str).apply(preprocess)
elif platform == "x":
    finfluencer_post_data['combined_text'] = finfluencer_post_data.apply(combine_x_metadata_text, axis=1)
    finfluencer_post_data["cleaned_text"] = finfluencer_post_data["combined_text"].astype(str).apply(preprocess)
    nonfinfluencer_post_data['combined_text'] = nonfinfluencer_post_data.apply(combine_x_metadata_text, axis=1)
    nonfinfluencer_post_data["cleaned_text"] = nonfinfluencer_post_data["combined_text"].astype(str).apply(preprocess)
else:
    raise ValueError(f"Platform {platform} is not supported.")

combined_post_data = pd.concat([finfluencer_post_data, nonfinfluencer_post_data], ignore_index=True)

# --------------------------------------------
# 3. Vectorise (bag-of-words with # and $ preserved)
# --------------------------------------------
vec = CountVectorizer(
    token_pattern=TOKEN_PATTERN, 
    ngram_range=(1,2),
    min_df=2
)  # ignore 1-offs
X = vec.fit_transform(combined_post_data["cleaned_text"])
terms = np.array(vec.get_feature_names_out())
y = combined_post_data["finfluencer"].values  # 1 = financial, 0 = non-financial

# convenience masks & counts
fin_mask = y == 1
non_mask = y == 0
fin_counts = X[fin_mask].sum(axis=0).A1
non_counts = X[non_mask].sum(axis=0).A1
total_counts = fin_counts + non_counts

def generate_word_cloud(combined_text: str, title: str) -> None:
    # Generate the word cloud
    wordcloud = WordCloud(width=800, height=400, background_color="white").generate(combined_text)

    # Display the word cloud
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.show()

# Generate word cloud for financial influencers
finfluencer_text = "\n".join(finfluencer_post_data["cleaned_text"].dropna())
generate_word_cloud(finfluencer_text, "Word Cloud for Financial Influencers")

# Generate word cloyd for non-financial influencers
nonfinfluencer_text = "\n".join(nonfinfluencer_post_data["cleaned_text"].dropna())
generate_word_cloud(nonfinfluencer_text, "Word Cloud for Non-Financial Influencers")

# --------------------------------------------
# 4-A. Approach 1  – Frequency & ratio table
# --------------------------------------------
ratio = (fin_counts + 1) / (non_counts + 1)  # +1 smoothing
df_ratio = (
    pd.DataFrame({
        "term": terms,
        "fin_freq": fin_counts,
        "non_freq": non_counts,
        "ratio": ratio
    })
    .sort_values("ratio", ascending=False)
    .reset_index(drop=True)
)

# --------------------------------------------
# 4-B. Approach 2  – χ² & Mutual-Information scores
# --------------------------------------------
chi2_scores, _ = chi2(X, y)
mi_scores = mutual_info_classif(X, y, discrete_features=True, random_state=42)

df_stats = pd.DataFrame({
    "term": terms,
    "chi2": chi2_scores,
    "mutual_info": mi_scores
}).sort_values("chi2", ascending=False).reset_index(drop=True)

# --------------------------------------------
# 4-C. Approach 3  – Class-based TF-IDF differential (c-TF-IDF)
# --------------------------------------------
# Step 1: term frequency (per class)
tf_fin  = fin_counts / fin_counts.sum()
tf_non  = non_counts / non_counts.sum()

# Step 2: IDF across classes (here, 'documents' = 2 class corpora)
# IDF = log( (#classes) / (1 + #classes_containing_term) )
classes_with_term = ((fin_counts > 0).astype(int) + (non_counts > 0).astype(int))
idf = np.log(2 / (classes_with_term + 1e-9))

# Step 3: c-TF-IDF
ctf_fin = tf_fin * idf
ctf_non = tf_non * idf
diff_score = ctf_fin - ctf_non    # positive = finance-heavy, negative = non-finance

df_ctfidf = (
    pd.DataFrame({
        "term": terms,
        "cTFIDF_fin": ctf_fin,
        "cTFIDF_non": ctf_non,
        "diff": diff_score
    })
    .query("cTFIDF_fin > 0")       # must appear in financial class
    .sort_values("diff", ascending=False)
    .reset_index(drop=True)
)

# -------------------------------------------------------------------------
# 5. Combine search terms from various analysis and remove duplicated terms
# -------------------------------------------------------------------------
search_terms_ratio = df_ratio.iloc[:TOP_N_TERMS]["term"].tolist()
search_terms_stats = df_stats.iloc[:TOP_N_TERMS]["term"].tolist()
search_terms_ctfidf = df_ctfidf.iloc[:TOP_N_TERMS]["term"].tolist()

combined_terms = list(set(search_terms_ratio + search_terms_stats + search_terms_ctfidf))
print(len(combined_terms))
print(combined_terms)

# --------------------------------------------
# 6. Save results for inspection
# --------------------------------------------
df_ratio.to_csv(os.path.join("ai_population/data", PROJECT_NAME, EXECUTION_DATE, "search_term_analysis_ratio.csv"), index=False)
df_stats.to_csv(os.path.join("ai_population/data", PROJECT_NAME, EXECUTION_DATE, "search_term_analysis_stats.csv"), index=False)
df_ctfidf.to_csv(os.path.join("ai_population/data", PROJECT_NAME, EXECUTION_DATE, "search_term_analysis_ctfidf.csv"), index=False)

# Evaluate Identified Search Terms on Holdout Set

In [ ]:
if platform == "tiktok":
    search_terms = ['#personalfinancetip', 'ticker symbol', '#financialfreedom', 'treasury', 'stock', '#invest', '#investingbeginner', '#finance', '#investingtips', 'trump', 'portfolio', '#financetips', 'ai', '#stocks', '#investing', 'buy share', '#stockstowatch', '#financetok', 'bond market', 'report earn', '#stockstobuy', 'tariff', 'stock market', '#personalfinance', 'etf', '#investing101', 'company', '#stockmarket', '#money', 'revenue', 'investor', '#financialliteracy']
elif platform == "x":
    search_terms = ['$vix', 'ebitda', 'bullish', 'fed', 'watchlist', 'ai', 'roce', '$spx', '$spy', '#elliottwave', 'reversal', 'bearish', 'stock', 'trade', '$qqq', 'tradinglounge', 'fomc', 'trader', '#stockmarket', '$coin']
else:
    raise ValueError(f"Platform {platform} is not supported.")
print(len(search_terms))

# Tiktok Old Search Terms
# search_terms = ["stocks", "stock market", "stock picks", "sp 500", "top stock", "underrated stocks", "stockstowatch", "stockstobuy", "invest", "invest follow", "investing stocks", "investingtips", "investing101", "investingbeginner", "follow trades", "daytrading", "option traders", "tariffs", "company", "business", "inflation", "interest rates", "ticker symbol", "wall street", "cash flow", "millennial money", "money finance", "finance investing", "financial advice" "financetips", "financetok", "financialfreedom", "financialliteracy", "united states", "donald trump", "news", "ai"]

# X Old Search Terms
# search_terms = ["stocks", "stock market", "investing", "finance stock market", "dividends", "market cap", "stock watchlist", "spx spy", "es spx", "spy qqq", "dia djia", "trading", "entry price", "profit per share", "elliot wave trading", "fastest momentum", "momentum system", "trading zone", "short float", "jerome powell", "rate cuts", "bitcoin", "traderinsights", "smallaccounttrading", "abnormal returns", "tradinglounge", "stockstotrade", "optiontrading", "marketsurge",]

In [ ]:
# ---------------------------
# 0. Load hold-out posts
# ---------------------------
# Load validation finfluencer and non-finfluencer lists
holdout_profiles = pd.read_csv(os.path.join("ai_population/data", PROJECT_NAME, EXECUTION_DATE, "ground_truth_holdout_set.csv"))
ground_truth_finfluencers = holdout_profiles[holdout_profiles["finfluencer"] == 1]["account_id"].tolist()
ground_truth_nonfinfluencers = holdout_profiles[holdout_profiles["finfluencer"] == 0]["account_id"].tolist()
print(f"Number of Financial Influencers: {len(ground_truth_finfluencers)}")
print(f"Number of Non-Financial Influencers: {len(ground_truth_nonfinfluencers)}")

# Split posts into finfluencer and non-finfluencer categories
post_data = pd.read_csv(os.path.join("ai_population/data", PROJECT_NAME, EXECUTION_DATE, "ground_truth_profile_posts.csv"))
finfluencer_post_data = post_data[post_data["account_id"].isin(ground_truth_finfluencers)].reset_index(drop=True)
finfluencer_post_data["finfluencer"] = 1
nonfinfluencer_post_data = post_data[post_data["account_id"].isin(ground_truth_nonfinfluencers)].reset_index(drop=True)
nonfinfluencer_post_data["finfluencer"] = 0

# ---------------------------
# 1. Preprocess hold-out posts
# ---------------------------
# Combine profile metadata and post metadata for each post
if platform == "tiktok":
    finfluencer_post_data['combined_text'] = finfluencer_post_data.apply(combine_tiktok_metadata_text, axis=1)
    finfluencer_post_data["cleaned_text"] = finfluencer_post_data["combined_text"].astype(str).apply(preprocess)
    nonfinfluencer_post_data['combined_text'] = nonfinfluencer_post_data.apply(combine_tiktok_metadata_text, axis=1)
    nonfinfluencer_post_data["cleaned_text"] = nonfinfluencer_post_data["combined_text"].astype(str).apply(preprocess)
elif platform == "x":
    finfluencer_post_data['combined_text'] = finfluencer_post_data.apply(combine_x_metadata_text, axis=1)
    finfluencer_post_data["cleaned_text"] = finfluencer_post_data["combined_text"].astype(str).apply(preprocess)
    nonfinfluencer_post_data['combined_text'] = nonfinfluencer_post_data.apply(combine_x_metadata_text, axis=1)
    nonfinfluencer_post_data["cleaned_text"] = nonfinfluencer_post_data["combined_text"].astype(str).apply(preprocess)
else:
    raise ValueError(f"Platform {platform} is not supported.")

combined_post_data = pd.concat([finfluencer_post_data, nonfinfluencer_post_data], ignore_index=True)

tokens_per_post = combined_post_data["cleaned_text"].str.split().apply(set).tolist()
y_hold = combined_post_data["finfluencer"].values

# ---------------------------
# 2. Per-term TP / FP / FN
# ---------------------------
results = []
for term in search_terms:
    term_lower = term.lower()
    # Boolean mask: does each post contain *exactly* this token?
    hit = np.array([term_lower in tok_set for tok_set in tokens_per_post])
    
    tp = np.sum(hit & (y_hold == 1))
    fp = np.sum(hit & (y_hold == 0))
    fn = np.sum(~hit & (y_hold == 1))
    
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall    = tp / (tp + fn) if tp + fn else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
    
    results.append({
        "term": term,
        "TP": tp, 
        "FP": fp, 
        "FN": fn,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3)
    })

df_eval = pd.DataFrame(results)

# ---------------------------
# 3. Rank by discriminative power
#     (primary: precision; tie-break by F1)
# ---------------------------
df_ranked = (
    df_eval.sort_values(
        by=["f1"], ascending=[False]
    ).reset_index(drop=True)
)

# ---------------------------
# 4. Combined-set evaluation
# ---------------------------
TOP_K = 10
top_terms = df_ranked["term"].iloc[:TOP_K].tolist()
print(f"\nEvaluating combined filter: top {TOP_K} terms → ...")

# union-of-hits mask
combined_hit = np.zeros(len(combined_post_data), dtype=bool)
for term in top_terms:
    term_lower = term.lower()
    combined_hit |= np.array([term_lower in tok_set for tok_set in tokens_per_post])

tp_c = np.sum(combined_hit & (y_hold == 1))
fp_c = np.sum(combined_hit & (y_hold == 0))
fn_c = np.sum(~combined_hit & (y_hold == 1))

precision_c = tp_c / (tp_c + fp_c) if tp_c + fp_c else 0.0
recall_c    = tp_c / (tp_c + fn_c) if tp_c + fn_c else 0.0
f1_c        = 2 * precision_c * recall_c / (precision_c + recall_c) if precision_c + recall_c else 0.0

print(f"""
=== Combined-Set Metrics (top-{TOP_K})  ===
Precision : {precision_c:.3f}
Recall    : {recall_c:.3f}
F1-score  : {f1_c:.3f}
------------------------------------------
TP={tp_c}, FP={fp_c}, FN={fn_c}, Total posts={len(combined_post_data)}
""")

# ---------------------------
# 5. Save all term metrics
# ---------------------------
df_ranked.to_csv(os.path.join("ai_population/data", PROJECT_NAME, EXECUTION_DATE, "search_term_evaluation_results.csv"), index=False)
